In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement RMS Normalization forward pass for 1D input vectors. Given an input tensor of shape [N] where N is the number of elements, compute the normalized output using a scalar scale (<code>gamma</code>) and shift (<code>beta</code>) parameter.
</p>

<p>
  RMS Normalization computes:
  $$
  \begin{align}
  \text{rms} &= \sqrt{\frac{1}{N} \sum_{i=1}^{N} x_i^2 + \epsilon} \\
  \hat{x}_i &= \frac{x_i}{\text{rms}} \\
  y_i &= \gamma \hat{x}_i + \beta
  \end{align}
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> tensor</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  input = [1.0, 2.0, 3.0, 4.0]  (N=4)
        gamma = 1.0
        beta = 0.0
        eps = 1e-5
Output: output = [0.36514813, 0.73029625, 1.0954444, 1.4605925 ]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  input = [1.0, 2.0, 3.0]  (N=3)
        gamma = 1.0
        beta = 0.0
        eps = 1e-5
Output: output = [0.46290955, 0.9258191, 1.3887286]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>N</code> ≤ 100,000</li>
  <li><code>eps</code> = 1e-5</li>
  <li>-100.0 ≤ input values ≤ 100.0</li>
  <li>0.1 ≤ gamma ≤ 10.0</li>
  <li>-10.0 ≤ beta ≤ 10.0</li>

  <li>Performance is measured with <code>N</code> = 100,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers
extern "C" void solve(const float* input, float gamma, float beta, float* output, int N,
                      float eps) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    gamma: cute.Float32,
    beta: cute.Float32,
    output: cute.Tensor,
    N: cute.Int32,
    eps: cute.Float32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input, gamma, beta are tensors on the GPU
@jax.jit
def solve(input: jax.Array, gamma: jax.Array, beta: jax.Array, N: int, eps: float) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    gamma: Float32,
    beta: Float32,
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    eps: Float32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    output: torch.Tensor,
    N: int,
    eps: float,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, output are tensors on the GPU
def solve(input: torch.Tensor, gamma: float, beta: float, output: torch.Tensor, N: int, eps: float):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="RMS Normalization", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        input: torch.Tensor,
        gamma: float,
        beta: float,
        output: torch.Tensor,
        N: int,
        eps: float,
    ):
        assert input.shape == output.shape == (N,)
        assert input.dtype == output.dtype
        assert input.device == output.device

        # RMSNorm: compute root mean square (without mean-centering)
        rms = torch.sqrt(torch.mean(input**2) + eps)  # shape: scalar

        # Normalize
        normalized = input / rms  # shape: [N]

        # Scale and shift
        output.copy_(gamma * normalized + beta)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "gamma": (ctypes.c_float, "in"),
            "beta": (ctypes.c_float, "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
            "eps": (ctypes.c_float, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 4
        input = torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype)
        gamma = 1.0
        beta = 0.0
        output = torch.empty(N, device="cuda", dtype=dtype)
        eps = 1e-5
        return {
            "input": input,
            "gamma": gamma,
            "beta": beta,
            "output": output,
            "N": N,
            "eps": eps,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # basic_small
        N = 3
        tests.append(
            {
                "input": torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=dtype),
                "gamma": 1.0,
                "beta": 0.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # single_feature
        N = 1
        tests.append(
            {
                "input": torch.tensor([5.0], device="cuda", dtype=dtype),
                "gamma": 2.0,
                "beta": -1.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # all zeros
        N = 4
        tests.append(
            {
                "input": torch.zeros(N, device="cuda", dtype=dtype),
                "gamma": 1.0,
                "beta": 0.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # negative numbers
        N = 5
        tests.append(
            {
                "input": torch.tensor([-1.0, -2.0, -3.0, -4.0, -5.0], device="cuda", dtype=dtype),
                "gamma": 1.0,
                "beta": 0.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # different gamma/beta
        N = 3
        tests.append(
            {
                "input": torch.tensor([0.0, 1.0, 2.0], device="cuda", dtype=dtype),
                "gamma": 0.5,
                "beta": -1.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # large values
        N = 8
        tests.append(
            {
                "input": torch.empty(N, device="cuda", dtype=dtype).uniform_(-100.0, 100.0),
                "gamma": 1.5,
                "beta": 0.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        # large N
        N = 2000
        tests.append(
            {
                "input": torch.empty(N, device="cuda", dtype=dtype).uniform_(-100.0, 100.0),
                "gamma": 1.3,
                "beta": 0.0,
                "output": torch.empty(N, device="cuda", dtype=dtype),
                "N": N,
                "eps": 1e-5,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 100000
        return {
            "input": torch.empty(N, device="cuda", dtype=dtype).uniform_(-10.0, 10.0),
            "gamma": 1.5,
            "beta": 0.0,
            "output": torch.empty(N, device="cuda", dtype=dtype),
            "N": N,
            "eps": 1e-5,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
